# Getting started

This notebook covers the **mechanics** of talking to the exchange — connecting, looking at state, sending and cancelling orders, and listening to events.

It does **not** tell you what to trade, how to price anything, or what the rules of the game imply. Figuring that out is your job.

**Before you run anything:**
1. The exchange server must be running (admin will have it open at some `http://host:8000`).
2. You have your own API key (8 chars-ish string the admin gave you).
3. From this notebook's parent folder, `pip install requests websockets`.

## 1. Connect

In [10]:
# Adjust to wherever this notebook lives relative to the repo
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from sdk.client import GameClient

URL = "http://192.168.50.167:8000"
API_KEY = "intern2-KEVD"  # intern2

c = GameClient(URL, API_KEY)

## 2. What's the game right now?

These are plain HTTP GETs. Use them whenever you want a fresh snapshot.

In [11]:
c.game_state()
# Returns roughly:
# {
#   'phase': 'waiting' | 'running' | 'settled',
#   'reveals': [list of positive ints drawn so far],
#   'running_sum': int,
#   'remaining_seconds': float | None,
#   'duration': int, 'reveal_interval': int,
#   'distribution': str, 'distribution_params': {...},
#   'instruments': { symbol: {...} },
#   'settle_prices': { symbol: int }  # only populated when settled
# }

{'phase': 'settled',
 'paused': False,
 'reveals': [6, 4, 4, 6, 7, 4],
 'running_sum': 31,
 'remaining_seconds': None,
 'duration': 300,
 'reveal_interval': 30,
 'instruments': {'A': {'symbol': 'A',
   'settlement': 'identity',
   'settlement_params': {},
   'tick_size': 1,
   'position_limit': 100}},
 'settle_prices': {'A': 31},
 'latency': {'processing_ms': 5, 'public_feed_ms': 20},
 'rate_limit': {'max_per_second': 20, 'lockout_seconds': 3.0},
 'fees': {'maker_per_lot': 0.5, 'taker_per_lot': 0.5}}

In [12]:
c.instruments()
# { symbol: { 'settlement': str, 'settlement_params': {...}, 'tick_size': int, 'position_limit': int } }

{'A': {'symbol': 'A',
  'settlement': 'identity',
  'settlement_params': {},
  'tick_size': 1,
  'position_limit': 100}}

## 3. Look at the order book

In [13]:
c.book("A")             # one instrument
# { 'symbol': 'A', 'bids': [{'price', 'qty'}, ...], 'asks': [...], 'ts_ns': int }

# Other forms:
# c.book()              # all instruments at once -> { symbol: book }
# c.book("A", depth=20) # more levels

{'symbol': 'A', 'bids': [], 'asks': [], 'ts_ns': 1779134550109332100}

## 4. Send orders

All order calls return synchronously with whatever happened: the order's status, and any immediate fills.

**Heads up:** orders are only accepted while `phase == 'running'`. Outside that you get HTTP 403.

In [14]:
# Limit order — rest on the book if it doesn't cross
result = c.buy("A", price=50, qty=5)
result
# {
#   'order': {
#     'order_id': 17,
#     'symbol': 'A',
#     'side': 'buy',
#     'price': 50,
#     'qty': 5,
#     'remaining': 5,           # how much still resting (0 if fully filled)
#     'status': 'open',          # 'open' | 'partial' | 'filled' | 'cancelled' | 'rejected'
#     'type': 'limit',
#     'tif': 'gtc',              # 'gtc' (default) | 'ioc' | 'fok'
#     ...
#   },
#   'trades': []                 # if you crossed, this lists each fill
# }

HTTPError: 403 Client Error: Forbidden for url: http://192.168.50.167:8000/api/order

In [ ]:
# Limit sell
c.sell("A", price=52, qty=3)

# Market order — takes whatever's on the book (up to qty). Markets are IOC
# by definition: any unfilled remainder is cancelled, never rests.
c.buy_market("A", qty=2)
c.sell_market("A", qty=2)

# Time-in-force (TIF) on limit orders:
#   - "gtc" (default): rest on the book until matched or cancelled.
#   - "ioc": match what you can at this price right now; cancel the rest.
#   - "fok": all-or-nothing. If the full qty can't fill immediately, the
#           order is born CANCELLED with zero fills (no partial fill).
c.buy("A", price=50, qty=5, tif="ioc")     # limit IOC
c.buy_ioc("A", price=50, qty=5)            # ...or use the wrapper

c.buy("A", price=50, qty=5, tif="fok")     # all-or-nothing
c.buy_fok("A", price=50, qty=5)            # ...or use the wrapper

# Optional: tag with your own id for tracking. The server echoes it back
# on fills and acks; it never has to be unique to the exchange.
c.buy("A", price=49, qty=1, client_order_id="my-tag-001")

### What can make an order fail?
- HTTP **400** — bad price (non-positive, not a multiple of `tick_size`), invalid side/type, or invalid `tif`.
- HTTP **401** — bad/missing API key.
- HTTP **403** — game not running, **or** the order would push your position past `position_limit` for that instrument.
- HTTP **404** — unknown symbol.

These come back from `c.buy/sell/...` as a `requests.HTTPError`. Wrap calls in try/except in your bot.

**A successful submission can still produce no fills:**
- A `MARKET` or `tif="ioc"` order that finds nothing to match returns with `status: "cancelled"` and `trades: []`. Not an HTTP error — it just didn't trade.
- A `tif="fok"` order whose full qty cannot be filled immediately returns the same way: `status: "cancelled"`, `trades: []`.

Always inspect `result["order"]["status"]` after submitting.

## 5. Your open orders and positions

In [ ]:
c.my_orders()         # all open orders across all symbols
# c.my_orders("A")    # filter by symbol

In [ ]:
c.positions()
# {
#   'trader': 'intern1',
#   'positions': { 'A': 5, ... },   # net per symbol (positive = long, negative = short)
#   'cash': { 'A': -250, ... },     # cash per symbol
#   'trades': 7                     # total fill count across all symbols
# }

## 6. Cancel

In [ ]:
# Place an order and grab its id
r = c.buy("A", price=40, qty=10)
oid = r["order"]["order_id"]

c.cancel(oid)              # cancel one order — server finds which book it's in
# c.cancel_all()           # cancel ALL my open orders (every symbol)
# c.cancel_all("A")        # cancel all my open orders on one symbol

## 6b. Modify a resting order

`c.modify(order_id, price=..., qty=...)` changes a resting GTC limit order in place. Pass whichever of `price` / `qty` you want to update (at least one).

**Queue-priority rule (this matters):** real exchanges only let you keep your spot in the FIFO queue if you make your order *less aggressive*. Specifically:

| Change | Same `order_id`? | Keeps FIFO priority? |
|---|---|---|
| `qty` strictly *down* at same price | yes | **yes** — your spot in line is preserved |
| `qty` *up* at same price | yes | **no** — re-stamped, goes to the back of the queue |
| `price` change (any direction) | yes | **no** — moved to a new level; if the new price now crosses, **it executes immediately** |
| Modify a partially filled order | yes | qty rules apply to remaining qty: `new_qty` must exceed already-filled |

Only **GTC limit** orders are modifiable. You cannot modify a market / IOC / FOK order (they don't rest). The response field `kept_priority: bool` tells you what happened.

In [ ]:
# Place a resting order to play with
r = c.buy("A", price=40, qty=10)
oid = r["order"]["order_id"]

# Reduce qty at same price -> keeps priority
res = c.modify(oid, qty=6)
print("kept_priority:", res["kept_priority"])   # True

# Increase qty -> loses priority (back of the queue at price 40)
res = c.modify(oid, qty=12)
print("kept_priority:", res["kept_priority"])   # False

# Change price -> loses priority; if new price crosses the book, this fills
res = c.modify(oid, price=41)
print("kept_priority:", res["kept_priority"])   # False
print("filled this much from the modify:", sum(t["qty"] for t in res["trades"]))

# Clean up
c.cancel(oid)

## 7. Listen for events (the live feed)

Polling `c.book(...)` works but is wasteful. Subscribe to the WebSocket feed instead. One connection delivers everything: your fills, the public trade tape, book deltas, reveals, and game-state changes.

Set callback functions, then call `c.start()`. It runs the WS in a background thread; your main code can keep going.

In [ ]:
def on_fill(msg):
    # YOUR fill (private). Includes 'liquidity' (maker/taker) and 'counterparty'.
    print(f"FILL  {msg['symbol']:3s} {msg['side']:4s} {msg['qty']} @ {msg['price']}  "
          f"liq={msg['liquidity']}  vs {msg['counterparty']}")

def on_trade(msg):
    # ANY trade on the tape (public). No trader names by design.
    print(f"TRADE {msg['symbol']:3s} {msg['qty']} @ {msg['price']}  aggressor={msg['aggressor']}")

def on_book(msg):
    bb = msg['bids'][0]['price'] if msg['bids'] else None
    ba = msg['asks'][0]['price'] if msg['asks'] else None
    print(f"BOOK  {msg['symbol']:3s} bid={bb} ask={ba}")

def on_reveal(msg):
    print(f"REVEAL #{msg['index']} = {msg['value']}  running_sum={msg['running_sum']}")

def on_ack(msg):
    # Order accepted. Note: status can be 'cancelled' right out of the gate
    # for IOC/FOK orders that couldn't fill — always check msg['status'].
    print(f"ACK   order_id={msg['order_id']} symbol={msg['symbol']} "
          f"status={msg['status']} tif={msg.get('tif')}")

def on_cancel_ack(msg):
    print(f"CXL   order_id={msg['order_id']} success={msg['success']}")

def on_modify_ack(msg):
    # Modify succeeded. kept_priority=True iff it was a pure qty-down.
    print(f"MOD   order_id={msg['order_id']} px={msg['price']} rem={msg['remaining']} "
          f"kept_priority={msg['kept_priority']}")

def on_game_state(msg):
    print(f"STATE phase={msg.get('phase')} reveals={len(msg.get('reveals') or [])}")

def on_settlement(msg):
    print(f"SETTLE prices={msg.get('prices')}  pnl={msg.get('pnl')}")

c.on_fill = on_fill
c.on_trade = on_trade
c.on_book = on_book
c.on_reveal = on_reveal
c.on_ack = on_ack
c.on_cancel_ack = on_cancel_ack
c.on_modify_ack = on_modify_ack
c.on_game_state = on_game_state
c.on_settlement = on_settlement

c.start()   # safe to call multiple times — only the first does anything